In [1]:
import os
import sys
import torch
import logging
import traceback
import numpy as np
from pprint import pprint

from runner import *
from utils.logger import setup_logging
from utils.arg_helper import parse_arguments, get_config
torch.set_printoptions(profile='full')

#===============================================================
# import datajoint as dj
# schema = dj.schema('xor-neuron')  # Change this
#
# @schema
# class Sample(dj.Manual):
#   definition = """
#     sample_id : int unsigned
#     """
# @schema
# class SampleComputed(dj.Computed):
#   definition = """
#     -> Sample
#     """
#   def make(self, key):
#     sample_id = key['sample_id']
#     # Call main function to start computation
#     main(sample_id)
#     self.insert1(key)  # Just to mark it as completed
#===============================================================


In [2]:
from easydict import EasyDict as edict
target_config_path = 'config/xor_neuron_mlp_mnist.yaml'

try:
    from runner import *
    from utils.logger import setup_logging
    from utils.arg_helper import get_config
except ImportError as e:
    raise e

args = edict()
args.config_file = target_config_path
args.test = False
args.log_level = 'INFO'
args.comment = 'notebook_run'

sample_id=1
config = get_config(args.config_file, sample_id="{:03d}".format(sample_id))

np.random.seed(config.seed)
torch.manual_seed(config.seed)
torch.cuda.manual_seed_all(config.seed)
config.use_gpu = config.use_gpu and torch.cuda.is_available()


log_file = os.path.join(config.save_dir, "log_exp_{}.txt".format(config.run_id))


logger = setup_logging(args.log_level, log_file)

logger.info("Writing log file to {}".format(log_file))
logger.info("Exp instance id = {}".format(config.run_id))
logger.info("Exp comment = {}".format(args.comment))

print(f"🚀 正在启动 Runner: {config.runner}")
print(f"📂 结果将保存到: {config.save_dir}")


try:
    runner = eval(config.runner)(config)

    if not args.test:
        # --- [Step 1] Pretrain ---
        print("\n--- [Step 1] Pretraining (InnerNet) ---")
        
        num_cells = config.model.num_cell_types
        print(f"💡 检测到配置需要预训练 {num_cells} 种细胞类型...")

        for i in range(num_cells):
            print(f"   > 正在预训练细胞类型: {i}")
            runner.pretrain(i)

        # --- [Step 2] Phase 1 ---
        print("\n--- [Step 2] Training Phase 1 (Joint Training) ---")
        runner.train_phase1()

        # --- [Step 3] Phase 2 ---
        print("\n--- [Step 3] Training Phase 2 (Freeze Activation) ---")
        runner.train_phase2()

        # --- [Step 4] Testing ---
        print("\n--- [Step 4] Final Testing ---")
        runner.test()
        
    else:
        # --- Local Test ---
        print("\n--- Local Testing Mode ---")
        runner.test_local()

    print("\n所有任务执行完毕！")

except Exception:
    logger.error(traceback.format_exc())
    print("\n运行出错，错误堆栈如下：")
    traceback.print_exc()

INFO  | 2025-12-20 20:29:06,257 | 2257388409.py             | line 31   : Writing log file to /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/log_exp_394113.txt
INFO  | 2025-12-20 20:29:06,258 | 2257388409.py             | line 32   : Exp instance id = 394113
INFO  | 2025-12-20 20:29:06,258 | 2257388409.py             | line 33   : Exp comment = notebook_run


🚀 正在启动 Runner: XorNeuronRunner
📂 结果将保存到: /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06

--- [Step 1] Pretraining (InnerNet) ---
💡 检测到配置需要预训练 10 种细胞类型...
   > 正在预训练细胞类型: 0
Pretrain Loss @ epoch 0001 = 0.0030136367717275487
Pretrain Loss @ epoch 0002 = 0.0015392872274982065
Pretrain Loss @ epoch 0003 = 0.0014745556716626348
Pretrain Loss @ epoch 0009 = 0.001390702359590824
Pretrain Loss @ epoch 0010 = 0.0013611268971432048
Pretrain Loss @ epoch 0011 = 0.0012622918080011816
Pretrain Loss @ epoch 0014 = 0.0012377132441242704
Pretrain Loss @ epoch 0015 = 0.0011892746539178386
Pretrain Loss @ epoch 0016 = 0.0011431331971199117
Pretrain Loss @ epoch 0019 = 0.001126572060050783
Pretrain Loss @ epoch 0020 = 0.0010583882098022886
Pretrain Loss @ epoch 0023 = 0.0010395513867814709
Pretrain Loss @ epoch 0029 = 0.0009519074720249159
Pretrain Loss @ epoch 0036 = 0.0008922694092706858
Pretrain Loss @ epoch 0037 = 0.0008903861218765017
Pretrain 

/home/yc/repo/xor_neuron_yoon/utils/train_helper.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_snapshot = torch.load(file_name, map_location=device)


Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained1.pth to cuda
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained2.pth to cuda
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained3.pth to cuda
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained4.pth to cuda
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained5.pth to cuda
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_pretrained6.pth to cuda
Loaded mod

100%|██████████| 100/100 [00:03<00:00, 29.12it/s]
INFO  | 2025-12-20 20:40:43,023 | inference_runner.py       | line 200  : Avg. Validation Loss = 2.3062964415550233 +- 0
INFO  | 2025-12-20 20:40:43,030 | inference_runner.py       | line 215  : Current Best Validation Loss = 2.3062964415550233
/home/yc/.local/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
100%|██████████| 100/100 [00:03<00:00, 31.05it/s]
INFO  | 2025-12-20 20:41:42,058 | inference_runner.py       | line 200  : Avg. Validation Loss = 0.3580993713438511 +- 0
INFO  | 2025-12-20 20:41:42,077 | inference_runne

KeyboardInterrupt: 

In [3]:
runner.train_phase2()


Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_phase1.pth to cuda


100%|██████████| 100/100 [00:03<00:00, 29.41it/s]
INFO  | 2025-12-20 22:43:07,406 | inference_runner.py       | line 390  : Avg. Validation Loss = 2.36911256313324 +- 0
INFO  | 2025-12-20 22:43:07,416 | inference_runner.py       | line 405  : Current Best Validation Loss = 2.36911256313324
100%|██████████| 100/100 [00:02<00:00, 34.46it/s]
INFO  | 2025-12-20 22:43:44,227 | inference_runner.py       | line 390  : Avg. Validation Loss = 0.35979178816080093 +- 0
INFO  | 2025-12-20 22:43:44,240 | inference_runner.py       | line 405  : Current Best Validation Loss = 0.35979178816080093
100%|██████████| 100/100 [00:03<00:00, 32.71it/s]
INFO  | 2025-12-20 22:44:22,044 | inference_runner.py       | line 390  : Avg. Validation Loss = 0.23683029033243655 +- 0
INFO  | 2025-12-20 22:44:22,061 | inference_runner.py       | line 405  : Current Best Validation Loss = 0.23683029033243655
100%|██████████| 100/100 [00:02<00:00, 37.39it/s]
INFO  | 2025-12-20 22:44:58,032 | inference_runner.py       | lin

KeyboardInterrupt: 

In [4]:
runner.test()

/home/yc/repo/xor_neuron_yoon/utils/train_helper.py:38: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model_snapshot = torch.load(file_name, map_location=device)


InnerNetData
train
/home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06
Loaded model from /home/yc/repo/xor_neuron_yoon/exp/xor_neuron_mlp_mnist/ComplexNeuronMLP_001_mnist_2025-Dec-20-20-29-06/model_snapshot_best_phase2.pth to cuda


100%|██████████| 100/100 [00:03<00:00, 32.68it/s]
INFO  | 2025-12-20 22:53:54,382 | inference_runner.py       | line 522  : Test Accuracy = 0.9742 +- 0


0.9742